# PDPA Assistant — Measure accuracy

This notebook gives your project **real numbers**:

1. **Retrieval:** how often the correct page is found
2. **Off-topic:** whether unrelated questions get "I don't know"
3. **Answer accuracy:** whether the written answer contains the key fact (e.g. "3 calendar days")

**Before/after experiment:** run the whole notebook twice — once with `IMPROVED = False` (the original version)
and once with `IMPROVED = True` — and compare. The improved version adds two fixes found during testing:

- a prompt rule to **copy numbers exactly** (the "24 hours" vs "3 calendar days" error)
- a **similarity threshold**: refuse in code, without asking the AI, when no chunk is similar enough

Results are saved to separate files.

**Before you run it:**

1. Run `pdpa_assistant.ipynb` at least once (it saves the embeddings in `pdpa_db`, which this notebook reuses).
2. Open `data/test_questions.csv` in Excel. For each **normal** question:
    - open the PDF named in `correct_file` and find the page that answers it
    - type that page number in `correct_page` (several pages: separate with `;`, e.g. `12;13`)
    - if the answer is in a different file, fix `correct_file` too
    - if **another document also answers it**, list those pages in `also_correct` as `file:page`,
      e.g. `key_concepts.pdf:143` (several: separate with `;`)
3. Add your own questions if you like. Use `normal` or `off-topic` in the `type` column.
4. Save it as **CSV** (not .xlsx).

**Why find the pages yourself?** The test checks the AI against the truth. If the AI wrote the answer key,
you'd be testing it against itself.

## Settings

Use the **same** model names as `pdpa_assistant.ipynb` — retrieval only works if questions and chunks
are embedded by the same model.

In [ ]:
from pathlib import Path

# The project folder (works whether you open this notebook from notebooks/ or the project root)
PROJECT_FOLDER = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_FOLDER = PROJECT_FOLDER / "pdpa_db"
TEST_FILE = PROJECT_FOLDER / "data" / "test_questions.csv"
(PROJECT_FOLDER / "results").mkdir(exist_ok=True)
# Before/after switch: False = original version, True = improved version
IMPROVED = False

STRICT_NUMBERS = IMPROVED                          # prompt rule: copy numbers exactly
MIN_SIMILARITY = 0.69 if IMPROVED else None        # refuse in code below this best-match similarity
RESULTS_FILE = PROJECT_FOLDER / "results" / f"test_results_{'improved' if IMPROVED else 'baseline'}.csv"

EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3.2"
TOP_K = 4

## Load the saved vector index

We **load** the saved embeddings instead of re-embedding every chunk — much faster.

In [ ]:
import json
import numpy as np
import ollama

EMBEDDINGS = np.load(DB_FOLDER / "embeddings.npy")
with open(DB_FOLDER / "chunks.json", encoding="utf-8") as f:
    all_chunks = json.load(f)
print(f"Loaded {len(all_chunks)} chunks")


def embed_texts(texts):
    response = ollama.embed(model=EMBED_MODEL, input=texts)
    return response["embeddings"]


def retrieve(question, top_k=TOP_K):
    q = np.array(embed_texts([question])[0], dtype=np.float32)
    q = q / np.linalg.norm(q)
    scores = EMBEDDINGS @ q
    best = np.argsort(scores)[::-1][:top_k]
    return [{"text": all_chunks[i]["text"], "source": all_chunks[i]["source"],
             "page": all_chunks[i]["page"], "similarity": round(float(scores[i]), 3)} for i in best]

## Load the test questions

The cell warns you about any normal question that still has no page filled in — those are skipped.

In [ ]:
import csv

with open(TEST_FILE, encoding="utf-8-sig") as f:
    tests = list(csv.DictReader(f))

normal = [t for t in tests if t["type"].strip().lower() == "normal"]
off_topic = [t for t in tests if t["type"].strip().lower() == "off-topic"]

print(f"{len(tests)} questions: {len(normal)} normal, {len(off_topic)} off-topic")

missing = [t["question"] for t in normal if not t["correct_page"].strip()]
for q in missing:
    print("  No page filled in yet (will be skipped):", q)

---
## Test 1 — Retrieval accuracy

For each normal question, we retrieve the top chunks and check:
**is the correct file AND page among them?** That's a *hit*.

The percentage of hits is your **retrieval score** (in the field it's called *hit rate at k*, here k = 4).

In [ ]:
def parse_pages(text):
    """'12;13' → {12, 13}"""
    return {int(p) for p in text.replace(",", ";").split(";") if p.strip().isdigit()}


def parse_also(text):
    # 'key_concepts.pdf:143;pdpa_act.pdf:65' -> {('key_concepts.pdf', 143), ('pdpa_act.pdf', 65)}
    pairs = set()
    for item in (text or "").split(";"):
        if ":" in item:
            file, page = item.rsplit(":", 1)
            if page.strip().isdigit():
                pairs.add((file.strip(), int(page)))
    return pairs


results = []
normal_best = []          # best similarity score for each PDPA question (used in the similarity report)
for t in normal:
    pages = parse_pages(t["correct_page"])
    if not pages:
        continue
    correct = {(t["correct_file"].strip(), p) for p in pages} | parse_also(t.get("also_correct", ""))
    found = retrieve(t["question"])
    normal_best.append(found[0]["similarity"])
    hit = any((c["source"], c["page"]) in correct for c in found)
    retrieved = "; ".join(f"{c['source']} p{c['page']}" for c in found)
    results.append({"question": t["question"], "type": "normal",
                    "expected": f"{t['correct_file']} p{t['correct_page']}",
                    "retrieved": retrieved, "result": "HIT" if hit else "MISS"})
    print(f"{'HIT ' if hit else 'MISS'}  {t['question']}")
    if not hit:
        print(f"        expected {t['correct_file']} p{t['correct_page']}, got: {retrieved}")

hits = sum(r["result"] == "HIT" for r in results)
if results:
    print(f"\nRETRIEVAL SCORE: {hits} of {len(results)} correct ({hits / len(results):.0%})")
else:
    print("No scored questions yet — fill in correct_page in test_questions.csv")

**Look at every MISS.** Common reasons:
- the answer really is on another page too (update your answer key),
- the question uses very different words from the document,
- the chunk containing the answer is split badly.

Write down 1–2 reasons — these are great interview stories.

---
## Test 2 — Off-topic questions

These need the AI to write an answer, so this step uses the chat model and takes a bit longer.

A **pass** means the answer is **only** the refusal ("I don't have information on that...").
An answer that half-answers and then refuses counts as a **fail** — an earlier, looser check missed this.

In the improved version, questions whose best match is below `MIN_SIMILARITY` are refused in code,
without asking the AI.

In [ ]:
REFUSAL = "I don't have information on that in my PDPA documents."
STRICT_RULE = ("5. Copy numbers, deadlines, amounts and time periods exactly as they are written "
               "in the sources. Never change, convert or estimate them.")


def build_prompt(question, found):
    context = "\n\n".join(f"[{n}] ({c['source']}, page {c['page']})\n{c['text']}"
                           for n, c in enumerate(found, start=1))
    return f"""You are an assistant that explains Singapore's Personal Data Protection Act (PDPA).

Rules:
1. Answer using ONLY the numbered sources below.
2. After each fact, cite the source number in square brackets, for example [1].
3. If the sources do not contain the answer, reply exactly: "{REFUSAL}"
4. Use simple, clear language that a small business owner can understand.
{STRICT_RULE if STRICT_NUMBERS else ''}

Sources:
{context}

Question: {question}
Answer:"""


def generate(prompt):
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}],
                           options={"temperature": 0})
    return response["message"]["content"]


def answer_question(question):
    # Returns (answer, best similarity, refused_by_threshold)
    found = retrieve(question)
    best = found[0]["similarity"]
    if MIN_SIMILARITY is not None and best < MIN_SIMILARITY:
        return REFUSAL, best, True
    return generate(build_prompt(question, found)), best, False


def is_refusal_only(answer):
    text = answer.strip().strip('"').strip().lower().replace("’", "'")
    starts = text.startswith("i don't have information") or text.startswith("i do not have information")
    return starts and len(text) <= len(REFUSAL) + 15


refused = 0
offtopic_best = []
for t in off_topic:
    answer, best, by_threshold = answer_question(t["question"])
    offtopic_best.append(best)
    passed = is_refusal_only(answer)
    refused += passed
    results.append({"question": t["question"], "type": "off-topic", "expected": "refusal only",
                    "retrieved": answer[:150].replace("\n", " "), "result": "PASS" if passed else "FAIL"})
    how = "  (refused by threshold)" if by_threshold else ""
    print(f"{'PASS' if passed else 'FAIL'}  {t['question']}   [best similarity {best}]{how}")
    if not passed:
        print("        answer was:", answer[:200].replace("\n", " "))

if off_topic:
    print(f"\nOFF-TOPIC: {refused} of {len(off_topic)} correctly refused")

---
## Test 3 — Answer accuracy

Retrieval can be right while the **answer** is still wrong (like the "24 hours" mistake).
For each question with a `must_contain` value, we check the answer contains at least one of the expected
phrases (alternatives separated by `|`, upper/lower case ignored).

This is a simple automatic check, so **read the WRONG answers yourself**: occasionally a correct answer
uses different words than expected.

In [ ]:
def contains_any(answer, phrases):
    text = answer.lower()
    return any(p.strip().lower() in text for p in phrases.split("|") if p.strip())


checked = [t for t in normal if t.get("must_contain", "").strip()]
correct_answers = 0
for t in checked:
    answer, best, by_threshold = answer_question(t["question"])
    ok = contains_any(answer, t["must_contain"])
    correct_answers += ok
    results.append({"question": t["question"], "type": "answer", "expected": t["must_contain"],
                    "retrieved": answer[:200].replace("\n", " "), "result": "CORRECT" if ok else "WRONG"})
    print(f"{'CORRECT' if ok else 'WRONG  '}  {t['question']}")
    if not ok:
        print("          expected one of:", t["must_contain"])
        print("          answer was:", answer[:250].replace("\n", " "))
        if by_threshold:
            print(f"          (refused by threshold: best similarity {best} < {MIN_SIMILARITY} — threshold may be too high)")

if checked:
    print(f"\nANSWER ACCURACY: {correct_answers} of {len(checked)} ({correct_answers / len(checked):.0%})"
          f"   [improved: {IMPROVED}, model: {CHAT_MODEL}]")
else:
    print("No must_contain values in test_questions.csv yet")

---
## Similarity report — choosing the threshold

The threshold only works if real PDPA questions score **higher** than off-topic ones.
This compares the best-match similarity of both groups.

In [ ]:
if normal_best and offtopic_best:
    lowest_real = min(normal_best)
    highest_off = max(offtopic_best)
    print(f"Lowest best-match similarity for a PDPA question:   {lowest_real:.3f}")
    print(f"Highest best-match similarity for an off-topic one: {highest_off:.3f}")
    if highest_off < lowest_real:
        print(f"\nClear gap. Any threshold between them separates the two groups, e.g. {(lowest_real + highest_off) / 2:.2f}")
    else:
        print("\nOverlap: some off-topic questions score as high as real ones, so a threshold alone can't separate them.")
    print(f"Current MIN_SIMILARITY: {MIN_SIMILARITY}")

---
## Save the results

Saves every question and its result to the `results` folder — useful for your README and for comparing
before/after when you try improvements later.

In [ ]:
with open(RESULTS_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "type", "expected", "retrieved", "result"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} results to {RESULTS_FILE.name}")

print("\nSUMMARY")
print(f"  Version: {'improved (copy-numbers rule + threshold ' + str(MIN_SIMILARITY) + ')' if IMPROVED else 'original'} | model: {CHAT_MODEL}")
scored = hits + sum(r["result"] == "MISS" for r in results)
if scored:
    print(f"  Retrieval:       {hits} of {scored} correct pages")
if off_topic:
    print(f"  Off-topic:       {refused} of {len(off_topic)} refused")
if checked:
    print(f"  Answer accuracy: {correct_answers} of {len(checked)} answers contain the key fact")

## Write down your results

- **Retrieval**, **off-topic** and **answer accuracy** scores
- whether `IMPROVED` was False or True
- the similarity report numbers
- 1–2 reasons for any misses or wrong answers

**Before/after:** after the baseline run, set `IMPROVED = True` in the Settings cell, then
**Kernel → Restart** and **Run All** again. Compare the off-topic and answer-accuracy scores.

**Later upgrade:** change `CHUNK_SIZE` or `TOP_K` in `pdpa_assistant.ipynb`, rebuild the database,
run this notebook again and compare scores. That's your "before vs after" improvement story.